# QuetHoaDon · Week 4: huấn luyện, chấm điểm và so sánh B0 · M1a · M1b

Notebook chạy trên **Kaggle (GPU T4, Internet bật)**. Mỗi bước là một cell độc lập; đọc dòng **Mong đợi** ở đầu cell để biết kết quả đúng.

| Phần | Cell | Thời gian (T4) | Ghi chú |
|---|---|---|---|
| Cài đặt | 1 – 3 | 10 phút | chạy mỗi session mới |
| Huấn luyện M1b | 5 | 30 phút | bỏ qua nếu dùng checkpoint có sẵn (cell 4) |
| Huấn luyện M1a | 6 | 10 phút | |
| Chấm validation, B0, so sánh | 7 – 10 | 10 phút | |
| Chấm **test** (một lần) | 11 | 6 phút | chỉ sau khi chốt cấu hình |
| Kiểm tra ảnh trùng, seed, vẽ lỗi, lưu | 12 – 15 | 5 – 70 phút | |

**Dữ liệu cần gắn (Add Input):** dataset `quethoadon` (ảnh đã tiền xử lý + `token_dataset_v1`). Nếu muốn bỏ qua huấn luyện M1b, gắn thêm dataset chứa checkpoint (ví dụ `m1b-v2-result`, thư mục `runs/m1b_v2_.../best`).

**Quy ước:** tập test chỉ chấm một lần (cell 11), không dùng test để chọn cấu hình. Kết quả trong `/kaggle/working` **mất khi session tắt**: chạy cell 15 rồi tải zip về.

## 1. Cài đặt môi trường

In [ ]:
# CELL 1 - Cài detectron2 (LayoutXLM cần). Mong đợi: "detectron2 OK" (lần đầu khoảng 5-10 phút)
try:
    import detectron2
    print("detectron2 đã có:", detectron2.__version__)
except ImportError:
    !pip install -q 'git+https://github.com/facebookresearch/detectron2.git' --no-build-isolation 2>&1 | tail -3
    import detectron2; print("detectron2 OK", detectron2.__version__)

In [ ]:
# CELL 2 - Lấy mã nguồn từ GitHub. Mong đợi: một dòng commit của nhánh HoaiNam/m1b-training
import os
%cd /kaggle/working
if not os.path.exists('QuetHoaDon'):
    !git clone -q https://github.com/CaoLePhucThinh2203/QuetHoaDon.git
%cd /kaggle/working/QuetHoaDon
!git fetch -q origin && git checkout -q -f -B HoaiNam/m1b-training origin/HoaiNam/m1b-training && git log --oneline -1

In [ ]:
# CELL 3 [KIỂM THỬ] - Tìm dữ liệu rồi chạy unit test. Mong đợi: TOK, IMG đúng thư mục; "Ran 155 tests ... OK"
import glob, os
TOK = os.path.dirname(glob.glob('/kaggle/input/**/validation.jsonl', recursive=True)[0])
IMG = os.path.dirname(glob.glob('/kaggle/input/**/*.jpg', recursive=True)[0])
print("TOK =", TOK, "->", sorted(os.listdir(TOK)))
print("IMG =", IMG, "| số ảnh:", len(os.listdir(IMG)))
os.makedirs('/kaggle/working/eval', exist_ok=True)
!python -m unittest discover -s tests 2>&1 | grep -E "^(Ran|OK|FAILED)"

## 2. Huấn luyện

M1b = fine-tune toàn bộ LayoutXLM (cấu hình **v2**: trọng số lớp, chọn checkpoint theo macro F1 của 4 trường). M1a = encoder đóng băng, chỉ train lớp phân loại. Chỉ dùng `train` và `validation`; tập test không bị đọc trong lúc huấn luyện.

In [ ]:
# CELL 4 - Dùng checkpoint M1b đã có (thay cho cell 5). Mong đợi: in ra đường dẫn best/ có config.json + model.safetensors
found = glob.glob('/kaggle/input/**/runs/m1b_v2_*/best', recursive=True) or glob.glob('/kaggle/working/runs/m1b_v2_*/best')
CKPT2 = sorted(found)[-1] if found else None
print("CKPT2 =", CKPT2, "|", os.listdir(CKPT2) if CKPT2 else "chưa có: chạy cell 5")

In [ ]:
# CELL 5 - Huấn luyện M1b v2 (khoảng 30 phút). Mong đợi: các dòng [epoch n] ... *best*, cuối cùng status early_stopped hoặc finished
!python scripts/train_layoutxlm.py --token-dir "{TOK}" --image-dir "{IMG}" --out /kaggle/working/runs --name m1b_v2 \
    --class-weight sqrt_inverse --select-metric macro --epochs 20 --patience 5
CKPT2 = sorted(glob.glob('/kaggle/working/runs/m1b_v2_*/best'))[-1]; print("CKPT2 =", CKPT2)

In [ ]:
# CELL 6 - Huấn luyện M1a (encoder đóng băng, khoảng 10 phút). Mong đợi: "trainable params=6,921"
!python scripts/train_layoutxlm.py --token-dir "{TOK}" --image-dir "{IMG}" --out /kaggle/working/runs --name m1a \
    --freeze-encoder --lr 1e-3 --epochs 20 --patience 5 --select-metric macro
CKPT_A = sorted(glob.glob('/kaggle/working/runs/m1a_*/best'))[-1]; print("CKPT_A =", CKPT_A)

## 3. Chấm điểm trên validation

Bộ giải mã dùng chung cho mọi hệ thống: giải mã mềm, ghép số tiền và ngày bị OCR tách, luật từ khóa, luật ngày. Khớp **chịu lỗi OCR** là kết quả chính; khớp nghiêm ngặt cũng được lưu trong báo cáo JSON.

In [ ]:
# CELL 7 - Chấm M1b và M1a trên validation (khoảng 3 phút). Mong đợi: bảng F1 từng trường, "ORACLE", khoảng tin cậy bootstrap
CKPT_A = sorted(glob.glob('/kaggle/working/runs/m1a_*/best'))[-1] if glob.glob('/kaggle/working/runs/m1a_*/best') else None
!python scripts/evaluate_layoutxlm.py --checkpoint "{CKPT2}" --token-dir "{TOK}" --image-dir "{IMG}" --split validation --out /kaggle/working/eval/m1b_v2_val.json > /kaggle/working/eval/m1b_v2_val.txt
if CKPT_A:
    !python scripts/evaluate_layoutxlm.py --checkpoint "{CKPT_A}" --token-dir "{TOK}" --image-dir "{IMG}" --split validation --out /kaggle/working/eval/m1a_val.json > /kaggle/working/eval/m1a_val.txt
!grep -A6 "LENIENT, soft decoding + rules" /kaggle/working/eval/m1b_v2_val.txt

In [ ]:
# CELL 8 - Baseline B0 (khoảng 5-10 phút lần đầu vì trích embedding; lần sau dùng bộ nhớ đệm). Mong đợi: "selected k=32"
!python scripts/run_baseline_b0.py --token-dir "{TOK}" --image-dir "{IMG}" --cache /kaggle/working/b0_cache --out /kaggle/working/eval/b0_val.json > /kaggle/working/eval/b0_val.txt
!grep -E "^\[b0\]" /kaggle/working/eval/b0_val.txt

In [ ]:
# CELL 9 - Bảng so sánh B0 · M1a · M1b trên validation + kiểm định bootstrap theo cặp
!python scripts/compare_models.py --baseline B0 --report B0=/kaggle/working/eval/b0_val.json --report M1a=/kaggle/working/eval/m1a_val.json --report M1b=/kaggle/working/eval/m1b_v2_val.json --out /kaggle/working/eval/comparison_val.md

In [ ]:
# CELL 10 - Phân tích lỗi M1b theo nguyên nhân. Mong đợi: bảng nguyên nhân (OCR sai chữ chiếm đa số)
!python scripts/error_analysis.py --report /kaggle/working/eval/m1b_v2_val.json --out /kaggle/working/eval/error_analysis_m1b_val.md | head -14

## 4. Chấm tập test — chỉ một lần

Chạy cell này **sau khi** đã chốt checkpoint, `k` của B0 và bộ giải mã trên validation. Đổi `CONFIG_FROZEN = True` để xác nhận; không chỉnh gì theo kết quả test.

In [ ]:
# CELL 11 - Chấm test cho B0 · M1a · M1b (khoảng 6 phút)
CONFIG_FROZEN = False        # đổi thành True khi đã chốt cấu hình trên validation
assert CONFIG_FROZEN, "Chưa chốt cấu hình: không chấm test"
!python scripts/evaluate_layoutxlm.py --checkpoint "{CKPT2}" --token-dir "{TOK}" --image-dir "{IMG}" --split test --allow-test --out /kaggle/working/eval/m1b_v2_test.json > /kaggle/working/eval/m1b_v2_test.txt
!python scripts/evaluate_layoutxlm.py --checkpoint "{CKPT_A}" --token-dir "{TOK}" --image-dir "{IMG}" --split test --allow-test --out /kaggle/working/eval/m1a_test.json > /kaggle/working/eval/m1a_test.txt
!python scripts/run_baseline_b0.py --token-dir "{TOK}" --image-dir "{IMG}" --cache /kaggle/working/b0_cache --split test --allow-test --k 32 --out /kaggle/working/eval/b0_test.json > /kaggle/working/eval/b0_test.txt
!python scripts/compare_models.py --baseline B0 --report B0=/kaggle/working/eval/b0_test.json --report M1a=/kaggle/working/eval/m1a_test.json --report M1b=/kaggle/working/eval/m1b_v2_test.json --out /kaggle/working/eval/comparison_test.md
!python scripts/error_analysis.py --report /kaggle/working/eval/m1b_v2_test.json --out /kaggle/working/eval/error_analysis_m1b_test.md > /dev/null

## 5. Kiểm tra bổ sung

In [ ]:
# CELL 12 - Ảnh gần trùng giữa các tập và điểm khi loại chúng (khoảng 3 phút). Mong đợi: số hóa đơn val/test có bản trùng trong tập khác
!python scripts/find_duplicates.py --token-dir "{TOK}" --image-dir "{IMG}" --out /kaggle/working/eval/dup_report.json | head -14
!python scripts/rescore_without_duplicates.py --dups /kaggle/working/eval/dup_report.json --report B0=/kaggle/working/eval/b0_test.json --report M1a=/kaggle/working/eval/m1a_test.json --report M1b=/kaggle/working/eval/m1b_v2_test.json

In [ ]:
# CELL 13 - (Tùy chọn) Thêm seed để đo độ dao động; M1a mất ~10 phút mỗi seed, M1b ~30 phút mỗi seed
SEEDS = (1, 2)
for sd in SEEDS:
    !python scripts/train_layoutxlm.py --token-dir "{TOK}" --image-dir "{IMG}" --out /kaggle/working/runs --name m1a_s{sd} --seed {sd} --freeze-encoder --lr 1e-3 --epochs 20 --patience 5 --select-metric macro 2>&1 | tail -4
    ck = sorted(glob.glob(f'/kaggle/working/runs/m1a_s{sd}_*/best'))[-1]
    !python scripts/evaluate_layoutxlm.py --checkpoint "{ck}" --token-dir "{TOK}" --image-dir "{IMG}" --split test --allow-test --out /kaggle/working/eval/m1a_s{sd}_test.json > /dev/null
# M1b: thay "m1a_s" bằng "m1b_s", bỏ --freeze-encoder, đặt --lr 5e-5 --class-weight sqrt_inverse
runs = ",".join(["/kaggle/working/eval/m1a_test.json"] + [f"/kaggle/working/eval/m1a_s{s}_test.json" for s in SEEDS])
!python scripts/aggregate_seeds.py --system M1a={runs}

In [ ]:
# CELL 14 - (Tùy chọn) Vẽ 30 hóa đơn M1b lỗi trên test để xem tay. Mong đợi: "drew 30 receipts"
!python scripts/draw_errors.py --report /kaggle/working/eval/m1b_v2_test.json --token-dir "{TOK}" --image-dir "{IMG}" --split test --out /kaggle/working/error_images --limit 30

## 6. Lưu kết quả

Tải `week4_eval.zip` về máy (panel bên phải → Data → Output). File chỉ vài MB, **không gồm checkpoint**. Checkpoint nằm ở `/kaggle/working/runs/*/best` (khoảng 1,4 GB mỗi cái): tải riêng hoặc tạo Kaggle Dataset private từ thư mục `runs` nếu cần dùng lại.

In [ ]:
# CELL 15 - Nén kết quả nhỏ để tải về
!cd /kaggle/working && zip -q -r week4_eval.zip eval error_images runs/*/run.json b0_cache/*.pkl 2>/dev/null; ls -la /kaggle/working/week4_eval.zip